<a href="https://colab.research.google.com/github/Bvsrkrishna/Data-Analyst-and-Engineering/blob/main/ETL%20pipeline2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2 - ETL pipeline (Extract, Transform, Validate, Load)


In [ ]:
#@title Setup (run first) { display-mode: "form" }
import sys, os
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False  #@param {type:"boolean"}
if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/clv_project")   # files survive between sessions
elif IN_COLAB:
    BASE = Path("/content/clv_project")                                                                              # lost when runtime resets
else:
    BASE = Path.cwd() / "clv_project"
BASE.mkdir(parents=True, exist_ok=True); os.chdir(BASE)
print("Working folder:", BASE)
REQUIRED = ['data/raw/transactions.csv', 'data/raw/customers.csv', 'data/raw/products.csv']
missing = [p for p in REQUIRED if not (BASE / p).exists()]

Working folder: /content/clv_project


In [ ]:
import os, sys, logging, time
import numpy as np, pandas as pd
from pathlib import Path


RAW, OUT = BASE / "data" / "raw", BASE / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
logging.basicConfig(force=True, level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("etl")
audit = []                                   # collected for the etl_audit table

def record(step, rows_in, rows_out, status="OK"):
    audit.append({"step": step, "rows_in": rows_in, "rows_out": rows_out, "status": status})
    log.info("%-22s in=%-8s out=%-8s %s", step, rows_in, rows_out, status)


### EXTRACT

In [ ]:
# ---------------------------------------------------------------- EXTRACT
def extract():
    tx = pd.read_csv(RAW / "transactions.csv")
    cu = pd.read_csv(RAW / "customers.csv")
    pr = pd.read_csv(RAW / "products.csv")
    record("extract", len(tx), len(tx))
    return tx, cu, pr
    tx

### TRANSFORM

In [ ]:
# ---------------------------------------------------------------- TRANSFORM
def transform(tx, cu, pr):
    n0 = len(tx)
    tx = tx.drop_duplicates().copy();                              record("drop_duplicates", n0, len(tx))
    n = len(tx); tx = tx.dropna(subset=["customer_id"]).copy();    record("drop_guest_rows", n, len(tx))
    tx["customer_id"] = tx.customer_id.astype(int)
    tx["order_date"] = pd.to_datetime(tx.order_date, errors="coerce")
    n = len(tx); tx = tx.dropna(subset=["order_date"]);            record("valid_dates", n, len(tx))

    returns = tx[tx.quantity <= 0]; returns.to_csv(OUT / "returns.csv", index=False)
    n = len(tx); tx = tx[tx.quantity > 0].copy();                  record("remove_returns", n, len(tx))

    n = len(tx)
    tx = tx[(tx.unit_price > 0) & tx.discount.between(0, 1)]
    tx = tx[tx.customer_id.isin(cu.customer_id) & tx.product_id.isin(pr.product_id)].copy()
    record("referential_checks", n, len(tx))

    # impute missing country from the customer master
    cmap = cu.set_index("customer_id").country
    tx["country"] = tx.country.fillna(tx.customer_id.map(cmap)).fillna("Unknown").str.strip().str.title()
    tx["line_revenue"] = (tx.quantity * tx.unit_price * (1 - tx.discount)).round(2)

    cu = cu.copy(); cu["signup_date"] = pd.to_datetime(cu.signup_date)
    cu["country"] = cu.country.fillna("Unknown"); cu["gender"] = cu.gender.fillna("Other")
    return tx.reset_index(drop=True), cu, pr

### VALIDATE

In [ ]:
# ---------------------------------------------------------------- VALIDATE
def validate(tx):
    checks = {"no_null_keys": tx[["order_id", "customer_id", "product_id"]].notna().all().all(),
              "positive_revenue": (tx.line_revenue > 0).all(),
              "no_dupes": not tx.duplicated().any(),
              "dates_in_range": tx.order_date.between("2024-01-01", "2025-12-31").all()}
    for k, v in checks.items(): log.info("check %-18s %s", k, "PASS" if v else "FAIL")
    if not all(checks.values()): raise ValueError(f"Data quality failed: {checks}")
    record("validate", len(tx), len(tx))

### FEATURES (RFM snapshot)

In [ ]:
# ---------------------------------------------------------------- FEATURES (RFM snapshot)
def build_rfm(tx):
    o = tx.groupby(["order_id", "customer_id", "order_date"], as_index=False).line_revenue.sum()
    snap = o.order_date.max() + pd.Timedelta(days=1)
    rfm = o.groupby("customer_id").agg(
        recency_days=("order_date", lambda s: (snap - s.max()).days),
        frequency=("order_id", "nunique"), monetary=("line_revenue", "sum"),
        first_order=("order_date", "min"), last_order=("order_date", "max")).reset_index()
    rfm["monetary"] = rfm.monetary.round(2); rfm["aov"] = (rfm.monetary / rfm.frequency).round(2)
    for col, asc in [("recency_days", False), ("frequency", True), ("monetary", True)]:
        rfm[col[0] + "_score"] = pd.qcut(rfm[col].rank(method="first", ascending=asc), 5, labels=False) + 1
    r, f = rfm.r_score, rfm.f_score
    rfm["rfm_segment"] = np.select(
        [(r >= 4) & (f >= 4), (r >= 3) & (f >= 3), (r >= 4) & (f <= 2), (r <= 2) & (f >= 3), (r <= 2) & (f <= 2)],
        ["Champions", "Loyal", "New / Promising", "At Risk", "Lost"], "Needs Attention")
    return rfm, o